## Ячейка 1 — Импорты и конфигурация

In [ ]:
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.chdir('/home/maki')

import re
import json
import time
import random
import operator
import numpy as np
from pathlib import Path
from collections import Counter

from huggingface_hub import login
HF_TOKEN = "HF"
login(token=HF_TOKEN)
print("HuggingFace авторизован")

CFG = {
    "n_3_from_7b"  : 4000,   # из verified_Qwen2.5-7B-Instruct
    "n_4_from_7b"  : 4000,
    "n_3_from_4b"  : 4000,    # из verified_Qwen3-4B-Instruct-2507
    "n_4_from_4b"  : 4000,
    
    # Модель
    "model_name"     : "./models/qwen3-4b-bnb-4bit",
    "output_dir"     : "./qwen3-4b-countdown/checkpoints",
    "lora_save_path" : "./qwen3-4b-countdown/lora",
    "gguf_save_path" : "./qwen3-4b-countdown-gguf",

    # Данные
    "dataset_name"   : "HuggingFaceTB/Countdown-Task-GOLD",
    "dataset_config" : "verified_Qwen3-4B-Instruct-2507",
    "val_split"      : 0.05,

    # Токенизатор
    "max_seq_length" : 1024,  

    # LoRA — минимальный rank для экономии памяти
    "lora_r"         : 16,
    "lora_alpha"     : 16,

    # Обучение — агрессивная экономия памяти
    "learning_rate"  : 2e-4,
    "batch_size"     : 2,     # минимум для 6GB
    "grad_accum"     : 8,    # эффективный batch = 16
    "num_epochs"     : 1,
    "warmup_ratio"   : 0.05,
    "lr_scheduler"   : "cosine",
    "optim"          : "adamw_8bit",
    "max_grad_norm"  : 1.0,
    "bf16"           : True,
    "weight_decay"   : 0.01,

    # Сохранение
    "save_steps"     : 40,
    "save_total_limit": 3,
    "logging_steps"  : 10,

    "seed"           : 42,
}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2))

In [ ]:
!ls

## Ячейка 2 — Вспомогательные функции

In [ ]:
def validate_equation(eq_str, nums, target):
    try:
        allowed = set("0123456789 +-*/().")
        if not set(eq_str) <= allowed:
            return False
        nums_in_eq = [int(x) for x in re.findall(r'\d+', eq_str)]
        nums_avail = sorted(nums)
        for n in sorted(nums_in_eq):
            if n not in nums_avail:
                return False
            nums_avail.remove(n)
        return abs(eval(eq_str) - target) < 1e-6
    except Exception:
        return False


def extract_equation(text):
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if not m:
        return None
    raw = m.group(1).strip()
    if '=' in raw:
        return raw.split('=')[0].strip()
    return raw


print("Вспомогательные функции загружены")

## Ячейка 3 — Загрузка датасета

In [ ]:
from datasets import load_dataset, Dataset

def load_and_prepare_dataset(cfg):
    print(f"Загружаем {cfg['dataset_name']} [{cfg['dataset_config']}]...")
    ds = load_dataset(
        cfg["dataset_name"],
        cfg["dataset_config"],
        split="train"
    )
    print(f"  Загружено: {len(ds)} примеров")

    # Валидация
    valid = []
    for ex in ds:
        msgs = ex["messages"]
        asst = next((m for m in msgs if m["role"] == "assistant"), None)
        if asst is None:
            continue
        eq = extract_equation(asst["content"])
        if eq and validate_equation(eq, ex["nums"], ex["target"]):
            valid.append({
                "target"  : ex["target"],
                "nums"    : list(ex["nums"]),
                "messages": list(ex["messages"]),
            })

    print(f"  Валидных: {len(valid)} (отфильтровано: {len(ds) - len(valid)})")

    dist = Counter(len(ex["nums"]) for ex in valid)
    print("  Распределение:")
    for k in sorted(dist):
        print(f"    {k} чисел: {dist[k]} ({dist[k]/len(valid)*100:.1f}%)")

    random.shuffle(valid)
    val_size = int(len(valid) * cfg["val_split"])
    ds_all = Dataset.from_list(valid)
    split = ds_all.train_test_split(test_size=val_size, seed=cfg["seed"])

    print(f"  Train: {len(split['train'])} | Val: {len(split['test'])}")
    return split["train"], split["test"]


ds_train, ds_val = load_and_prepare_dataset(CFG)

## Ячейка 4 — Загрузка модели

In [ ]:
import torch
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

# Мониторинг памяти
def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved  = torch.cuda.memory_reserved()  / 1024**3
        print(f"  GPU: allocated={allocated:.2f}GB reserved={reserved:.2f}GB")

print("Загружаем Qwen3-8B...")
print_gpu_memory()

model, tokenizer = FastModel.from_pretrained(
    model_name     = CFG["model_name"],
    max_seq_length = CFG["max_seq_length"],
    load_in_4bit   = True,
    device_map     = "auto", 
    local_files_only = True,
)

print("После загрузки модели:")
print_gpu_memory()

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r                          = CFG["lora_r"],
    lora_alpha                 = CFG["lora_alpha"],
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = CFG["seed"],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nМодель загружена")
print(f"  LoRA trainable: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"  r={CFG['lora_r']}, alpha={CFG['lora_alpha']}")
print("После LoRA:")
print_gpu_memory()

## Ячейка 5 — Форматирование и обучение

In [ ]:
def formatting_func(examples):
    """
    Форматирует примеры через chat template Qwen3.
    Qwen3 использует ChatML: <|im_start|>role\ncontent<|im_end|>
    """
    if isinstance(examples["messages"][0], dict):
        all_messages = [examples["messages"]]
    else:
        all_messages = examples["messages"]

    texts = []
    for msgs in all_messages:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return texts

In [ ]:
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        step = state.global_step
        loss = logs.get("loss", "")
        lr   = logs.get("learning_rate", "")
        if loss:
            print(f"  step={step:>4} | loss={loss:.4f} | lr={lr:.2e}")


eff_batch   = CFG["batch_size"] * CFG["grad_accum"]
total_steps = len(ds_train) // eff_batch * CFG["num_epochs"]
print(f"Эффективный batch: {eff_batch}")
print(f"Всего шагов: {total_steps}")

training_args = SFTConfig(
    output_dir                  = CFG["output_dir"],
    num_train_epochs            = CFG["num_epochs"],
    per_device_train_batch_size = CFG["batch_size"],
    gradient_accumulation_steps = CFG["grad_accum"],
    learning_rate               = CFG["learning_rate"],
    optim                       = CFG["optim"],
    weight_decay                = CFG["weight_decay"],
    max_grad_norm               = CFG["max_grad_norm"],
    warmup_ratio                = CFG["warmup_ratio"],
    lr_scheduler_type           = CFG["lr_scheduler"],
    bf16                        = CFG["bf16"],
    max_seq_length              = CFG["max_seq_length"],
    packing                     = True,
    eval_strategy               = "no",
    save_strategy               = "steps",
    save_steps                  = CFG["save_steps"],
    save_total_limit            = CFG["save_total_limit"],
    logging_steps               = CFG["logging_steps"],
    report_to                   = "none",
    dataloader_num_workers      = 0,
    dataloader_pin_memory       = False,
    dataset_num_proc            = 0,
    seed                        = CFG["seed"],
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = ds_train,
    args             = training_args,
    formatting_func  = formatting_func,
    callbacks        = [PrintLossCallback()],
)

# Маскируем промпт — обучаем только на ответах
# Qwen3 использует ChatML формат
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("Trainer готов")

## Ячейка 6 — Запуск обучения

In [ ]:
print("\nНачинаем обучение...\n")
print_gpu_memory()

start  = time.time()
result = trainer.train()
elapsed = time.time() - start

print(f"\nОбучение завершено за {elapsed/60:.1f} минут")
print(f"  Train loss: {result.training_loss:.4f}")
print(f"  Шагов:      {result.global_step}")
print("\nПамять после обучения:")
print_gpu_memory()

## Ячейка 7 — Сохранение модели

In [ ]:
# Сохраняем LoRA адаптеры
model.save_pretrained(CFG["lora_save_path"])
tokenizer.save_pretrained(CFG["lora_save_path"])
print(f"LoRA сохранены: {CFG['lora_save_path']}")

# Сохраняем GGUF для использования как учителя через llama-server
print(f"\nСохраняем GGUF (q4_k_m)...")
model.save_pretrained_gguf(
    CFG["gguf_save_path"],
    tokenizer,
    quantization_method = "q4_k_m",
)
print(f"GGUF сохранён: {CFG['gguf_save_path']}")
print("\nГотово! Теперь можно использовать GGUF как учителя для генерации")
print("данных с 5-6 числами через llama-server.")